# 🧾 VisionDoc AI — Google Colab Quickstart

Fine-tune **Qwen2.5-VL** with **LoRA** for document intelligence, then evaluate and run inference — end to end, on a **free Colab T4 GPU**.

> **First:** set the runtime to a GPU — **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

Repo: https://github.com/Swapnil-byte-798/visiondoc-ai


## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import torch
ok = torch.cuda.is_available()
print("CUDA available:", ok, "| device:", torch.cuda.get_device_name(0) if ok else "NONE")
assert ok, "No GPU! Enable it via Runtime > Change runtime type > T4 GPU, then rerun."

## 2. Clone (or update) the repository

In [ ]:
import os
REPO = "/content/visiondoc-ai"
if os.path.isdir(REPO):
    !cd {REPO} && git pull -q            # re-run picks up the latest fixes
else:
    !git clone -q https://github.com/Swapnil-byte-798/visiondoc-ai.git {REPO}
%cd {REPO}
!git log --oneline -1

## 3. Install dependencies

Colab already ships a **CUDA build of PyTorch**, so we do **not** reinstall torch (that would be slow and can break CUDA). We only add the project's other deps and upgrade `transformers` to a version that ships the Qwen2.5-VL classes.

In [ ]:
# ~2-3 min. Ignore pip's dependency-resolver warnings about pre-installed Colab pkgs.
!pip install -q -U "transformers>=4.49.0" "accelerate>=0.34" "peft>=0.13" "datasets>=2.20" qwen-vl-utils bitsandbytes
!pip install -q evaluate rouge-score sacrebleu Levenshtein pymupdf pytesseract seaborn python-dotenv "pydantic>=2.7"
# OCR binary (optional — only needed for region-highlighting):
!apt-get -qq install -y tesseract-ocr > /dev/null
# Register the project as an editable package (deps already handled above):
!pip install -q -e . --no-deps
print("\n✅ Dependencies installed. If Colab shows a 'RESTART' button, click it, then continue from cell 4.")

## 4. Runtime settings

In [ ]:
import os
os.environ["WANDB_MODE"] = "offline"          # no Weights & Biases account needed
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/content/hf_cache"    # cache model + dataset downloads
print("Config we'll use:\n")
!sed -n '1,12p' configs/colab_t4.yaml

## 5. (Optional) Persist to Google Drive

Colab wipes local files when the session ends. Mount Drive if you want the downloaded model, dataset cache, and trained adapter to survive a disconnect.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/visiondoc-outputs
# # then point outputs there, e.g. train with:  --config configs/colab_t4.yaml  and later copy:
# # !cp -r outputs /content/drive/MyDrive/visiondoc-outputs/

## 6. Fine-tune with LoRA

This uses `configs/colab_t4.yaml`: Qwen2.5-VL-3B in **4-bit QLoRA + fp16**, on a small slice of **CORD** (receipts) so it finishes fast. The first run downloads the ~7 GB base model (a few minutes), then trains ~50 steps.

Only the LoRA adapters train — the backbone is frozen (you'll see e.g. *"trainable ≈ 0.3%"* in the log).

In [ ]:
!python -m training.train --config configs/colab_t4.yaml

## 7. Evaluate the fine-tuned adapter

In [ ]:
!python -m evaluation.evaluate --config configs/colab_t4.yaml \
    --adapter outputs/qwen2_5vl-cord-lora-colab/adapter \
    --split test --max-samples 100
# A self-contained HTML report is written under reports/ — open it from the Files panel.

## 8. Run inference (QA + structured extraction + confidence)

In [ ]:
from dataclasses import replace
from configs import load_config
from preprocessing import load_samples
from inference import DocumentPredictor

cfg = load_config("configs/colab_t4.yaml")
cfg = replace(cfg, inference=replace(cfg.inference,
              adapter_path="outputs/qwen2_5vl-cord-lora-colab/adapter"))

predictor = DocumentPredictor.from_config(cfg)      # loads base + LoRA adapter

sample = load_samples(cfg, "test", max_samples=1)[0]
res = predictor.answer(sample.image, sample.question)
print("Q :", sample.question)
print("A :", res.answer)
print("conf:", round(res.confidence, 3), "| latency(ms):", round(res.latency_ms, 1))

# Structured field extraction with a document-type template:
fields = predictor.extract(sample.image, doc_type="receipt")
print("\nExtracted fields:", fields["fields"])

## 9. (Optional) Research: base vs LoRA fine-tuned

Compares the frozen base model against your fine-tuned adapter on the same test split and writes `reports/comparison.csv` + a bar chart.

In [ ]:
!python -m research.compare --config configs/colab_t4.yaml \
    --adapter outputs/qwen2_5vl-cord-lora-colab/adapter --max-samples 80

## 10. (Optional) Launch the Streamlit dashboard via a tunnel

Colab can't expose ports directly, so we tunnel port 8501 with **localtunnel** (no signup). Uncomment to run. The printed `https://*.loca.lt` URL opens the dashboard; if it asks for a password, it's the VM's public IP printed by the last command.

In [ ]:
# import os
# os.environ["VISIONDOC_ADAPTER_PATH"] = "outputs/qwen2_5vl-cord-lora-colab/adapter"
# os.environ["VISIONDOC_CONFIG"] = "configs/colab_t4.yaml"
# get_ipython().system_raw("streamlit run app/streamlit_app.py --server.port 8501 --server.headless true &")
# !sleep 6 && npx --yes localtunnel --port 8501 &
# !curl -s https://loca.lt/mytunnelpassword   # <- tunnel password if prompted

---
### Notes & tips
- **DocVQA instead of CORD:** copy `configs/colab_t4.yaml`, set `data.dataset_name: docvqa`, `data.dataset_id: lmms-lab/DocVQA`, `data.dataset_subset: DocVQA` (larger download + longer training).
- **Out-of-memory?** lower `model.max_pixels` (e.g. `401408`), keep `gradient_accumulation_steps` high, and keep `per_device_train_batch_size: 1`.
- **Faster/real run:** raise `data.max_train_samples` (or set it to `null` for the full split) and `num_train_epochs`.
- **A100/L4 runtime:** you can switch `torch_dtype`/`bf16` back on and set `load_in_4bit: false` for full-precision LoRA.
